## 0. 준비 셀

In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [4]:
# 사이킷런을 이용해 당뇨병 데이터 다운로드
diabetes = fetch_openml(
    data_id=37,
    as_frame=True,
    parser="pandas"
)

# 특성 데이터 복사
df = diabetes.data.copy()

# 정답 추가 (0 = 음성, 1 = 양성)
df["target"] = diabetes.target.map({
    "tested_negative": 0,
    "tested_positive": 1
}).astype(int)

print(df.shape)
print(df.head(3))

(768, 9)
   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1


## 문제 1) 데이터 탐색
1. 전체 행 개수와 열 개수를 출력하세요.
2. 상위 5행을 출력하세요.
3. 각 열의 데이터 타입을 확인하세요.
4. `describe()`로 통계 정보를 확인하세요.
5. `target`의 값별 데이터 개수를 출력하세요.
6. 양성(1)과 음성(0)의 비율을 백분율로 출력하세요.

**분석 질문**

1. 전체 데이터에서 양성(1)의 비율은 몇 %인가요?
2. 입력 특성 중 숫자 범위가 가장 큰 특성과 가장 작은 특성은 무엇인가요?
3. 특성마다 숫자 범위가 다른 것이 KNN에 어떤 영향을 줄 수 있을까요?

In [9]:
print(df.shape)
print(df.head())
print(df.info())
df.describe()

(768, 9)
   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1
3     1    89    66    23    94  28.1  0.167   21       0
4     0   137    40    35   168  43.1  2.288   33       1
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    768 non-null    int64  
 2   pres    768 non-null    int64  
 3   skin    768 non-null    int64  
 4   insu    768 non-null    int64  
 5   mass    768 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   target  768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None


,preg,plas,pres,skin,insu,mass,pedi,age,target
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [19]:
print(df["target"].value_counts())
ratio = df["target"].value_counts()
ratio_negative = round(ratio [0] / (ratio[0]+ratio[1]) *100, 4)
ratio_positive = round(ratio [1] / (ratio[0]+ratio[1]) *100, 4)

print("양성 비율 :", ratio_positive)
print("음성 비율 :", ratio_negative)

target
0    500
1    268
Name: count, dtype: int64
양성 비율 : 34.8958
음성 비율 : 65.1042


- 전체 데이터 중 양성의 비율은 약 34.9퍼센트 정도 된다.
- 가장 범위가 큰 것은 인슐린수치로 0~846정도 되고, pedi(당뇨병 발병 유전적 위험도)가 0.078~2.42로 제일 작았다.
- KNN이 기본적으로 거리를 통해서 가까운 데이터 수가 많은걸로 양성인지 음성인지 판별하기 때문에 범위가 각기 다른게 정확도를 떨어뜨릴 수 있다고 생각한다. 따라서 스케일링을 통해 수치를 비슷하게 만들 필요가 있어보인다.

### 2. 결측치 탐색

데이터에서 누락된 측정값을 찾아보세요.

1. `isna().sum()`으로 결측치 개수를 출력하세요.
2. 준비 셀에서 안내한 5개 열의 0 개수를 각각 출력하세요.
3. 해당 5개 열에서 0을 `NaN`으로 변경하세요.
4. 다시 `isna().sum()`으로 결측치 개수를 확인하세요.
5. 결측치가 하나라도 포함된 행의 개수를 출력하세요.

**주의:** 이 단계에서는 결측치를 삭제하거나 다른 값으로 채우지 않습니다.

**분석 질문**

1. 처음 `isna()`를 실행했을 때 결측치가 발견되지 않은 이유는 무엇인가요?
2. 모든 열의 0을 `NaN`으로 변경하면 안 되는 이유는 무엇인가요?

In [20]:
df.isna().sum()

preg      0
plas      0
pres      0
skin      0
insu      0
mass      0
pedi      0
age       0
target    0
dtype: int64

In [28]:
zero_plas = len(df[(df["plas"]==0)])
zero_pres = len(df[(df["pres"]==0)])
zero_skin = len(df[(df["skin"]==0)])
zero_insu = len(df[(df["insu"]==0)])
zero_mass = len(df[(df["mass"]==0)])

print("혈당 수치가 0인 경우 :", zero_plas)
print("혈압 수치가 0인 경우 :", zero_pres)
print("피부 두께가 0인 경우 :", zero_skin)
print("인슐린 수치가 0인 경우 :", zero_insu)
print("BMI가 0인 경우 :", zero_mass)

혈당 수치가 0인 경우 : 5
혈압 수치가 0인 경우 : 35
피부 두께가 0인 경우 : 227
인슐린 수치가 0인 경우 : 374
BMI가 0인 경우 : 11


In [32]:
df["plas"] = df["plas"].replace(0, None)
df["pres"] = df["pres"].replace(0, None)
df["skin"] = df["skin"].replace(0, None)
df["insu"] = df["insu"].replace(0, None)
df["mass"] = df["mass"].replace(0, None)
df.isna().sum()

preg        0
plas        5
pres       35
skin      227
insu      374
mass       11
pedi        0
age         0
target      0
dtype: int64

In [35]:
len(df) - len(df.dropna())

376

- 결측치가 아닌 0으로 입력되었기 때문에 결측치로 발견되지 않았다.
- 임신횟수(preg)의 0은 정상적인 값이기 때문이며, 또한 타겟의 0도 음성을 나타내는 숫자이다.

### 특성과 정답 분리 및 데이터 분할

머신러닝을 학습하기 위한 데이터를 준비하세요.

1. `target`을 제외한 열들을 `X`에 저장하세요.
2. `target`을 `y`에 저장하세요.
3. `X`와 `y`의 shape를 출력하세요.
4. `train_test_split()`으로 데이터를 분리하세요.

**분할 조건**

- 학습 데이터 80%
- 테스트 데이터 20%
- `random_state=42`
- `stratify=y`
1. `X_train`, `X_test`, `y_train`, `y_test`의 shape를 출력하세요.
2. 학습 데이터와 테스트 데이터의 양성·음성 개수를 각각 출력하세요.

**분석 질문**

1. 학습 데이터와 테스트 데이터를 나누는 이유는 무엇인가요?
2. `stratify=y`를 사용하는 이유는 무엇인가요?
3. `random_state`를 지정하면 어떤 장점이 있나요?

In [41]:
X = df.drop(columns="target")
y = df["target"]
print(X.shape)
print(y.shape)
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(768, 8)
(768,)
(614, 8)
(154, 8)
(614,)
(154,)


- 학습데이터 테스트데이터를 분리하지 않으면 학습한 내용을 가지고 테스트를 해버리기 과적합이 방생한다.
- 학습데이터와 테스트데이터의 정답 비율을 똑같이 유지함으로 평가가 치우치지 않게 하기 위함이다.
- KNN의 K값을 바꿔가면서 어느 숫자를 사용해야 가장 적합한지 과적합 과소적합을 일으키지 않고 데이터를 보기위함이다.

### 4. 결측치 처리

3번에서 분리한 학습 데이터와 테스트 데이터의 결측치를 처리하세요.

**조건**

- 결측치는 각 특성의 중앙값(`median`)으로 채우세요.
- 중앙값은 학습 데이터에서만 계산하세요.
- 테스트 데이터에는 학습 데이터에서 계산한 중앙값을 사용하세요.
- 원본 데이터는 유지하고 결측치를 처리한 데이터를 별도의 변수에 저장하세요.

다음 작업을 수행하세요.

1. 학습 데이터에서 각 특성의 중앙값을 계산하세요.
2. 학습 데이터의 결측치를 중앙값으로 채우세요.
3. 테스트 데이터의 결측치도 동일한 중앙값으로 채우세요.
4. 처리 후 학습 데이터와 테스트 데이터의 결측치 개수를 `isna().sum()`으로 출력하세요.
5. 처리 전후 데이터의 행 개수를 비교하세요.

**분석 질문**

1. 결측치를 삭제하지 않고 중앙값으로 채우면 어떤 장점이 있나요?
2. 테스트 데이터의 중앙값을 따로 계산해서 사용하면 안 되는 이유는 무엇인가요?

In [49]:
median_X = X_train.median()
X_train_median = X_train.fillna(median_X)
X_test_median = X_test.fillna(median_X)

print("학습데이터 결측 갯수 :",X_train_median.isna().sum().sum())
print("테스트데이터 결측 갯수 :",X_test_median.isna().sum().sum())

학습데이터 결측 갯수 : 0
테스트데이터 결측 갯수 : 0


In [55]:
print("처리전 학습데이터 행 갯수 :", len(X_train))
print("처리전 테스트데이터 행 갯수 :", len(X_test))
print("처리후 학습데이터 행 갯수 :", len(X_train_median))
print("처리후 테스트데이터 행 갯수 :", len(X_test_median))

처리전 학습데이터 행 갯수 : 614
처리전 테스트데이터 행 갯수 : 154
처리후 학습데이터 행 갯수 : 614
처리후 테스트데이터 행 갯수 : 154


- 거의 절반이 결측치가 하나라도 있는 행이였기 때문에 많은 정보를 손실하는 셈이다. 따라서 부족한 데이터를 가지고 분석하기에는 너무많은 정보를 잃기 때문에 중앙값으로 처리하는 것이 그나마 이상적이다.
- 테스트 데이터의 중앙값을 계산하게되면 데이터가 누출될 수 있다.

### 5. 첫 번째 KNN 학습

결측치 처리가 완료된 데이터를 이용하여 KNN을 학습하세요.

**조건**

- `KNeighborsClassifier` 사용
- `n_neighbors=5`
- 스케일링하지 않은 데이터 사용
- 4번에서 결측치를 처리한 학습·테스트 데이터 사용

다음 작업을 수행하세요.

1. KNN 객체를 생성하세요.
2. `fit()`으로 학습하세요.
3. 학습 정확도를 출력하세요.
4. 테스트 정확도를 출력하세요.
5. `predict()`로 테스트 데이터의 결과를 예측하세요.
6. 예측 결과와 실제 정답을 각각 앞 10개씩 출력하세요.

**분석 질문**

1. KNN에서 k=5는 어떤 의미인가요?
2. 학습 정확도와 테스트 정확도는 각각 무엇을 의미하나요?
3. 학습 정확도와 테스트 정확도에 차이가 발생하는 이유는 무엇인가요?

In [64]:
model = KNeighborsClassifier()
model.fit(X_train_median, y_train)
score_train_median = model.score(X_train_median, y_train)
score_test_median = model.score(X_test_median, y_test)
print("학습 정확도 :", score_train_median)
print("테스트 정확도 :", score_test_median)
pred = model.predict(X_test_median)
pred_answer = pred[:10]
real_answer = y_test.values[:10]
print("예상 정답 :", pred_answer)
print("실제 정답 :", real_answer)

학습 정확도 : 0.8013029315960912
테스트 정확도 : 0.6753246753246753
예상 정답 : [1 0 0 0 0 0 0 1 0 1]
실제 정답 : [0 0 0 1 0 0 1 1 0 0]


- 한 행의 데이터가 그 주변에 있는 다섯 개의 데이터에서 가까운 데이터가 많은 정보를 통해 음성인지 양성인지를 판별한다.
- 학습 데이터는 말그대로 정답을 보고 데이터랑 비교해가며 모델이 학습하는 데이터의 정확도이며, 테스트 정확도는 학습데이터를 기반으로 처음보는 데이터를 보았을 때 예측한 답이 실제 답과 맞는지 정확도를 보는 것이다.
- 학습데이터의 정답을 기반으로 측정을 하기때문에 과적합 혹은 과소적합이 생길 수가 있다.

### 6. KNN 성능 평가

5번에서 학습한 KNN의 예측 결과를 평가하세요.

다음 항목을 출력하세요.

1. 혼동행렬 (`confusion_matrix`)
2. 정확도 (`accuracy_score`)
3. 정밀도 (`precision_score`)
4. 재현율 (`recall_score`)
5. F1 점수 (`f1_score`)
6. `classification_report`

이번 데이터는 이진 분류이므로 `0`은 음성, `1`은 양성입니다. 정밀도와 재현율은 양성(1)을 기준으로 계산하세요.

**분석 질문**

1. 실제 양성인데 음성으로 잘못 예측한 데이터(FN)는 몇 개입니까?
2. 실제 음성인데 양성으로 잘못 예측한 데이터(FP)는 몇 개입니까?
3. 정확도는 높지만 재현율은 낮은 상황이 발생할 수 있나요?
4. 당뇨병 양성을 놓치는 것을 줄이려면 어떤 평가 지표를 중요하게 봐야 할까요?

In [75]:
print("혼동행렬 :", confusion_matrix(y_test, pred))
print("정확도 :", accuracy_score(y_test, pred))
print("정밀도 :", precision_score(y_test, pred, pos_label=1))
print("재현율 :", recall_score(y_test, pred, pos_label=1))
print("F1점수 :", f1_score(y_test, pred))
print(classification_report(y_test, pred))

혼동행렬 : [[77 23]
 [27 27]]
정확도 : 0.6753246753246753
정밀도 : 0.54
재현율 : 0.5
F1점수 : 0.5192307692307693
              precision    recall  f1-score   support

           0       0.74      0.77      0.75       100
           1       0.54      0.50      0.52        54

    accuracy                           0.68       154
   macro avg       0.64      0.64      0.64       154
weighted avg       0.67      0.68      0.67       154



- 27개
- 23개
- 정확도는 전체 중에서 맞게 예측한 비율이기 때문에 거짓을 맞힌 것도 포함되지만, 재현율에서는 포함하지 않기 때문
- 재현율이다. 환자 중 절반밖에 찾지 못하기 때문에 재현율을 높여야 당뇨병 환자를 최대한 많이 치료하는데 도움이 되기 때문

### 7. k값에 따른 성능 비교

KNN의 k값을 변경하면서 학습 정확도와 테스트 정확도가 어떻게 변하는지 확인하세요.

실험할 k값:

`1, 3, 5, 7, 10, 15, 20, 30`

**조건**

- 반복문을 사용하세요.
- 스케일링하지 않은 데이터를 사용하세요.
- 4번에서 결측치를 처리한 데이터를 사용하세요.
- k값 외에는 동일한 조건을 유지하세요.

각 k값의 학습 정확도와 테스트 정확도를 출력하고 다음 표를 완성하세요.

마크다운 또는 코드셀에 주석으로 적어주시면 됩니다.
e.g. 
1 / 0.91 / 0.93 
3 / 0.92 / 0.94
이런식으로 ,,,,

| k | 학습 정확도 | 테스트 정확도 |
| --- | --- | --- |
| 1 |  |  |
| 3 |  |  |
| 5 |  |  |
| 7 |  |  |
| 10 |  |  |
| 15 |  |  |
| 20 |  |  |
| 30 |  |  |

**분석 질문**

1. 학습 정확도가 가장 높은 k값은 무엇인가요?
2. 테스트 정확도가 가장 높은 k값은 무엇인가요?
3. k가 작을 때 과대적합이 발생하기 쉬운 이유는 무엇인가요?
4. k가 지나치게 커지면 과소적합이 발생할 수 있는 이유는 무엇인가요?
5. 학습 정확도만 보고 k값을 선택해도 될까요?

In [83]:
k_values = [1,3,5,7,10,15,20,30]
result = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_median, y_train)
    train_score = model.score(X_train_median, y_train)
    test_score = model.score(X_test_median, y_test)
    result.append({
        "k": k,
        "학습 정확도" : train_score,
        "테스트 정확도" : test_score
    })

result = pd.DataFrame(result)
result.set_index("k")

,학습 정확도,테스트 정확도
k,,
1,1.000000,0.681818
3,0.842020,0.681818
5,0.801303,0.675325
7,0.785016,0.675325
10,0.798046,0.668831
15,0.791531,0.655844
20,0.788274,0.720779
30,0.765472,0.720779


- k=1
- k=20 또는 k=30
- k가 작을 때 적은 데이터를 보고 판단하기 때문에 경계가 복잡해진다.
- 너무 많은 데이터를 보고 비교하게 되면 근처의 값보다 더 많은 수를 가지고 데이터의 결과값을 예측하기 때문이다.
- 아니다! 테스트 정확도와 학습정확도 모두 고려해서 테스트의 정확도가 최고치일 때 학습정확도도 고려해서 정해야한다. 따라서 k=20이 적합해보인다.

### 8. StandardScaler 적용

이번에는 `StandardScaler`로 특성을 표준화한 뒤 동일한 KNN을 학습하세요.

**조건**

- 4번에서 결측치를 처리한 데이터 사용
- `StandardScaler` 사용
- 스케일러는 학습 데이터에만 `fit()`
- 학습 데이터와 테스트 데이터는 같은 스케일러로 변환
- KNN의 k값은 5로 고정

다음 작업을 수행하세요.

1. `StandardScaler` 객체를 생성하세요.
2. 학습 데이터로 `fit()` 하세요.
3. 학습 데이터와 테스트 데이터를 각각 `transform()` 하세요.
4. 변환된 학습 데이터를 DataFrame으로 만들고 `describe()`를 출력하세요.
5. 변환된 데이터로 KNN(k=5)을 학습하세요.
6. 학습 정확도와 테스트 정확도를 출력하세요.
7. 재현율을 출력하세요.

**분석 질문**

1. 표준화 후 각 특성의 평균과 표준편차는 대략 얼마인가요?
2. 5번에서 구한 스케일링 전 정확도와 비교하면 어떤 차이가 있나요?
3. 모델의 알고리즘과 k값은 그대로인데 정확도가 달라지는 이유는 무엇인가요?

In [87]:
scaler = StandardScaler()
scaler.fit(X_train_median)
X_train_scaled = scaler.transform(X_train_median)
X_test_scaled = scaler.transform(X_test_median)
df_scaled = pd.DataFrame(X_train_scaled, columns=X_train_median.columns)
df_scaled.describe()

,preg,plas,pres,skin,insu,mass,pedi,age
count,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02
mean,-6.943414e-17,-1.099374e-16,3.095606e-16,-3.471707e-17,-4.339634e-18,1.649061e-16,-1.099374e-16,-1.084908e-16
std,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00
min,-1.153338e+00,-2.190541e+00,-3.924957e+00,-2.480958e+00,-1.559139e+00,-2.089621e+00,-1.192094e+00,-1.045895e+00
25%,-8.513551e-01,-7.562208e-01,-6.636760e-01,-4.549828e-01,-2.249692e-01,-7.073644e-01,-7.042617e-01,-7.921693e-01
50%,-2.473897e-01,-1.558076e-01,-1.141979e-02,-4.766137e-03,-1.614373e-01,-7.070181e-03,-2.876344e-01,-3.692933e-01
75%,6.585583e-01,6.113872e-01,6.408364e-01,3.328964e-01,-9.790537e-02,5.942296e-01,4.903223e-01,6.456090e-01
max,3.980368e+00,2.579408e+00,4.065181e+00,7.874025e+00,7.703814e+00,5.081979e+00,5.610293e+00,4.028617e+00


In [91]:
model = KNeighborsClassifier()
model.fit(X_train_scaled, y_train)
scaled_train = model.score(X_train_scaled,y_train)
scaled_test = model.score(X_test_scaled,y_test)
print("학습 정확도 :", scaled_train)
print("테스트 정확도 :", scaled_test)
pred_scaled = model.predict(X_test_scaled)
recall_scaled = recall_score(y_test, pred_scaled, pos_label=1)
print("재현율 :", recall_scaled)

학습 정확도 : 0.8289902280130294
테스트 정확도 : 0.7532467532467533
재현율 : 0.6111111111111112


- 평균은 대략 0, 표준편차는 대략 1정도로 나온다.
- 전체적으로 크게 올라갔다.
- 처음에 비교했던 각 칼럼별 범위가 전부 달랐기 때문에 표준화를 통해서 거리를 비슷하게 맞춰줄 수 있었기 때문이다.

### 9. MinMaxScaler 적용 및 비교

이번에는 `MinMaxScaler`를 적용하고 KNN의 성능을 확인하세요.

**조건**

- 4번에서 결측치를 처리한 데이터 사용
- `MinMaxScaler` 사용
- 학습 데이터에만 `fit()`
- KNN의 k값은 5로 고정

다음 작업을 수행하세요.

1. `MinMaxScaler`로 학습·테스트 데이터를 변환하세요.
2. 변환된 학습 데이터의 `describe()`를 출력하세요.
3. 변환된 데이터로 KNN(k=5)을 학습하세요.
4. 학습 정확도와 테스트 정확도를 출력하세요.
5. 재현율을 출력하세요.
6. 세 가지 전처리 방법의 성능을 비교하세요.

| KNN 전처리 방법 | 학습 정확도 | 테스트 정확도 | 재현율 |
| --- | --- | --- | --- |
| 스케일링 없음 |  |  |  |
| StandardScaler |  |  |  |
| MinMaxScaler |  |  |  |

**분석 질문**

1. 테스트 정확도가 가장 높은 전처리 방법은 무엇인가요?
2. 재현율이 가장 높은 전처리 방법은 무엇인가요?
3. 정확도가 높은 방법과 재현율이 높은 방법이 다를 수도 있나요?
4. 스케일링을 적용한다고 해서 정확도가 반드시 높아지는 것은 아닌 이유는 무엇일까요?

In [93]:
minmax = MinMaxScaler()
minmax.fit(X_train_median)
X_train_minmax = minmax.transform(X_train_median)
X_test_minmax = minmax.transform(X_test_median)
df_minmax = pd.DataFrame(X_train_minmax, columns=X_train_median.columns)
df_minmax.describe()

,preg,plas,pres,skin,insu,mass,pedi,age
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,0.224660,0.459238,0.491225,0.239591,0.168320,0.291374,0.175246,0.206107
std,0.194950,0.209817,0.125256,0.096651,0.108045,0.139553,0.147127,0.197224
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.058824,0.300699,0.408163,0.195652,0.144033,0.192740,0.071715,0.050000
50%,0.176471,0.426573,0.489796,0.239130,0.150892,0.290389,0.132962,0.133333
75%,0.352941,0.587413,0.571429,0.271739,0.157750,0.374233,0.247327,0.333333
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [101]:
model = KNeighborsClassifier()
model.fit(X_train_minmax, y_train)
minmax_train = model.score(X_train_minmax, y_train)
minmax_test = model.score(X_test_minmax, y_test)
print("학습 데이터 :", minmax_train)
print("테스트 데이터 :", minmax_test)
pred_minmax = model.predict(X_test_minmax)
recall_minmax = recall_score(y_test, pred_minmax)
print("재현율 :", recall_minmax)

recall = recall_score(y_test, pred, pos_label=1)

df_compare = pd.DataFrame({
    "KNN 전처리 방법": ["스케일링 없음", "StandardScaler", "MinMaxScaler"],
    "학습 정확도": [score_train_median, scaled_train, minmax_train],
    "테스트 정확도": [score_test_median, scaled_test, minmax_test],
    "재현율": [recall, recall_scaled, recall_minmax]
})
df_compare

학습 데이터 : 0.8322475570032574
테스트 데이터 : 0.7337662337662337
재현율 : 0.5555555555555556


,KNN 전처리 방법,학습 정확도,테스트 정확도,재현율
0,스케일링 없음,0.801303,0.675325,0.500000
1,StandardScaler,0.828990,0.753247,0.611111
2,MinMaxScaler,0.832248,0.733766,0.555556


- 테스트 정확도가 제일 높은것은 StandardScaler이다.
- 재현율이 높은것도 StandardScaler이다.
- 다를 수도 있을 것이다. 분자 분모에 들어가는 값이 애초에 다르기 때문이다.
- 큰 범위 구간에 있는 것을 강제로 작게 만들기 때문에 값의 차이가 적어서 데이터의 값의 차이를 정확하게 식별하지 못할 수 있다.

### 10. 종합 — KNN 설정 선택 및 최종 분석

지금까지 실험한 결과를 이용하여 가장 적절하다고 생각하는 KNN 설정을 선택하세요.

**1단계 — 스케일링 방법 선택**

다음 중 하나를 선택하세요.

- 스케일링 없음
- StandardScaler
- MinMaxScaler

**2단계 — k값 실험**

선택한 스케일링 방법으로 다음 k값을 실험하세요.

`1, 3, 5, 7, 10, 15, 20, 30`

각 k값에 대해 다음 결과를 표로 정리하세요.

- 학습 정확도
- 테스트 정확도
- 재현율

**3단계 — 최종 설정 선택**

실험 결과를 비교하여 다음 두 가지를 결정하세요.

1. 사용할 스케일링 방법
2. 사용할 k값

단순히 정확도가 가장 높은 설정을 선택해도 되지만, 재현율이나 과대적합·과소적합 가능성을 고려하여 다른 설정을 선택해도 됩니다.

**4단계 — 선택한 설정으로 KNN 학습 및 평가**

선택한 설정으로 KNN을 학습하고 다음 결과를 출력하세요.

1. 학습 정확도
2. 테스트 정확도
3. 혼동행렬
4. 정밀도
5. 재현율
6. F1 점수

**5단계 — 최종 분석 보고서**

마크다운 셀에 다음 질문의 답변을 작성하세요.

1. 최종적으로 어떤 스케일링 방법과 k값을 선택했나요?
2. 해당 설정을 선택한 이유는 무엇인가요?
3. 스케일링 전후 KNN의 성능은 어떻게 달라졌나요?
4. k값이 달라지면서 정확도는 어떤 변화를 보였나요?
5. 정확도와 재현율 중 어떤 지표를 더 중요하게 고려했나요? 이유를 설명하세요.
6. 결측치 처리와 스케일링에서 학습 데이터에만 기준을 맞춰야 하는 이유는 무엇인가요?
7. 이번 과제를 진행하면서 머신러닝 전체 과정 중 가장 중요하다고 생각한 단계는 무엇인가요?

**참고:** 이번 과제는 아직 배우지 않은 검증 데이터·교차검증을 사용하지 않고, 동일한 테스트 데이터로 여러 설정의 결과를 비교합니다. 이는 수업 내용을 복습하기 위한 방식입니다. 실제 프로젝트에서는 설정 선택에 사용한 데이터와 최종 성능 평가에 사용할 테스트 데이터를 별도로 구분해야 합니다.

In [107]:
# 스케일링전

k_values = [1,3,5,7,10,15,20,30]
result = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_median, y_train)
    train_score = model.score(X_train_median, y_train)
    test_score = model.score(X_test_median, y_test)
    pred = model.predict(X_test_median)
    recall = recall_score(y_test, pred, pos_label=1)
    result.append({
        "k": k,
        "학습 정확도" : train_score,
        "테스트 정확도" : test_score,
        "재현율" : recall
    })

result = pd.DataFrame(result)
result.set_index("k")

,학습 정확도,테스트 정확도,재현율
k,,,
1,1.000000,0.681818,0.537037
3,0.842020,0.681818,0.518519
5,0.801303,0.675325,0.500000
7,0.785016,0.675325,0.481481
10,0.798046,0.668831,0.444444
15,0.791531,0.655844,0.444444
20,0.788274,0.720779,0.500000
30,0.765472,0.720779,0.481481


In [108]:
# StandardScaler

k_values = [1,3,5,7,10,15,20,30]
result = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    train_score = model.score(X_train_scaled, y_train)
    test_score = model.score(X_test_scaled, y_test)
    pred = model.predict(X_test_scaled)
    recall = recall_score(y_test, pred, pos_label=1)
    result.append({
        "k": k,
        "학습 정확도" : train_score,
        "테스트 정확도" : test_score,
        "재현율" : recall
    })

result = pd.DataFrame(result)
result.set_index("k")

,학습 정확도,테스트 정확도,재현율
k,,,
1,1.000000,0.720779,0.555556
3,0.846906,0.720779,0.611111
5,0.828990,0.753247,0.611111
7,0.817590,0.727273,0.555556
10,0.804560,0.740260,0.537037
15,0.802932,0.740260,0.555556
20,0.788274,0.759740,0.555556
30,0.788274,0.740260,0.518519


In [109]:
k_values = [1,3,5,7,10,15,20,30]
result = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_minmax, y_train)
    train_score = model.score(X_train_minmax, y_train)
    test_score = model.score(X_test_minmax, y_test)
    pred = model.predict(X_test_minmax)
    recall = recall_score(y_test, pred, pos_label=1)
    result.append({
        "k": k,
        "학습 정확도" : train_score,
        "테스트 정확도" : test_score,
        "재현율" : recall
    })

result = pd.DataFrame(result)
result.set_index("k")

,학습 정확도,테스트 정확도,재현율
k,,,
1,1.000000,0.701299,0.555556
3,0.871336,0.740260,0.592593
5,0.832248,0.733766,0.555556
7,0.830619,0.727273,0.555556
10,0.791531,0.733766,0.462963
15,0.799674,0.727273,0.537037
20,0.791531,0.733766,0.462963
30,0.786645,0.740260,0.500000


**5단계 — 최종 분석 보고서**

마크다운 셀에 다음 질문의 답변을 작성하세요.

1. 최종적으로 어떤 스케일링 방법과 k값을 선택했나요?
    
    StandardScaler의 k=5

2. 해당 설정을 선택한 이유는 무엇인가요?
    
    재현율이 제일 높은 값 중 테스트 정확도가 높은 것을 골랐다.

3. 스케일링 전후 KNN의 성능은 어떻게 달라졌나요?
    
    많이 높아졌다. 스케일링 하기 전보다 후에 특히 재현율이 0.5에서 0.61까지 크게 상승했다. 

4. k값이 달라지면서 정확도는 어떤 변화를 보였나요?
    
    학습정확도는 k=1일 때, 가장 높았으며 과적합의 형태를 보이며, k값이 높아질수록 학습정확도는 떨어진다.
    테스트정확도는 미세하게 오르락내리락하지만 점점 줄어드는 것처럼보인다. 

5. 정확도와 재현율 중 어떤 지표를 더 중요하게 고려했나요? 이유를 설명하세요.
    
    재현율을 더 중요하게 고려했다. 실제 당뇨병환자를 놓치지 않고 찾아야하기 때문이다.
    
6. 결측치 처리와 스케일링에서 학습 데이터에만 기준을 맞춰야 하는 이유는 무엇인가요?
    
    전처리 과정이기 때문에 테스트 문제를 미리 가져와 버리면 데이터가 누출되기 때문이다.

7. 이번 과제를 진행하면서 머신러닝 전체 과정 중 가장 중요하다고 생각한 단계는 무엇인가요?

    전처리 전과 전처리 후 그리고 스케일링까지 통해서 가장 적합한 방법과 적합한 K값을 찾아 모델에게 적용시키고 머신러닝을 통해 데이터를 분석하는 것이 가장 중요하다고 생각된다.